In [6]:
# packages and data
import numpy as np
import pandas as pd
from scipy import stats

BASE_DIR = "course/MidTerm"
P1 = f"{BASE_DIR}/problem1.csv"
ALPHA = 0.05


In [2]:
df1 = pd.read_csv(P1)
df1.head()
if "X" in df1.columns and np.issubdtype(df1["X"].dtype, np.number):
    x = df1["X"].to_numpy(dtype=float)
else:
    num_cols = [c for c in df1.columns if np.issubdtype(df1[c].dtype, np.number)]
    if len(num_cols) == 0:
        raise ValueError("problem1.csv no numerical column")
    x = df1[num_cols[0]].to_numpy(dtype=float)
x = x[~np.isnan(x)]
len(x), x[:5]


(1000,
 array([-0.0387906 , -0.04125756,  0.05322352, -0.09542313,  0.05626356]))

In [3]:
# Problem 1 (a)
mean_ = float(np.mean(x))
var_ = float(np.var(x, ddof=1))
skew_ = float(stats.skew(x, bias=False))
kurt_ = float(stats.kurtosis(x, fisher=False, bias=False))

desc = pd.DataFrame([{
    "Mean": mean_,
    "Variance": var_,
    "Skewness": skew_,
    "Kurtosis": kurt_,
}])
desc

,Mean,Variance,Skewness,Kurtosis
0,-0.000532,0.001007,-0.23987,3.689877


In [4]:
# Problem 1 (b)
reason = f"""
Based on part (a), skewness={skew_:.4f} and kurtosis={kurt_:.4f}.
Kurtosis is noticeably larger than 3 (normal's kurtosis), it indicates heavier tails,
so a Student-t distribution is typically more appropriate than a normal distribution.
""".strip()

print(reason)

Based on part (a), skewness=-0.2399 and kurtosis=3.6899.
Kurtosis is noticeably larger than 3 (normal's kurtosis), it indicates heavier tails,
so a Student-t distribution is typically more appropriate than a normal distribution.


In [5]:
# Problem 1 (c)
# Normal fit
mu_n = float(np.mean(x))
sig_n = float(np.std(x, ddof=1))

# Student-t fit
nu_t, mu_t, sig_t = stats.t.fit(x)
nu_t, mu_t, sig_t = float(nu_t), float(mu_t), float(sig_t)

fit_tbl = pd.DataFrame([{
    "Normal_mu": mu_n,
    "Normal_sigma": sig_n,
    "T_nu": nu_t,
    "T_mu": mu_t,
    "T_sigma": sig_t,
}])
fit_tbl

,Normal_mu,Normal_sigma,T_nu,T_mu,T_sigma
0,-0.000532,0.031728,13.711803,-0.000219,0.029299


In [11]:
# Problem 1 (c) explanation
ll_n = float(np.sum(stats.norm.logpdf(x, loc=mu_n, scale=sig_n)))
ll_t = float(np.sum(stats.t.logpdf(x, df=nu_t, loc=mu_t, scale=sig_t)))

aic_n = 2*2 - 2*ll_n  # k=2: mu,sigma
aic_t = 2*3 - 2*ll_t  # k=3: nu,mu,sigma

cmp = pd.DataFrame([
    {"Model": "Normal", "LogLik": ll_n, "AIC": aic_n},
    {"Model": "StudentT", "LogLik": ll_t, "AIC": aic_t},
]).sort_values("AIC")

cmp


,Model,LogLik,AIC
1,StudentT,2037.066788,-4068.133577
0,Normal,2032.122283,-4060.244567


In [12]:
print("Since Student-t distribution clearcly has a smaller AIC which means it is more appropriate to model the data")

Since T-distribution clearcly has a smaller AIC which means it is more appropriate to model the data


In [13]:
# Problem 1 (d) alpha = 0.05 VaR
def var_metrics_from_quantile(q: float, mean_value: float) -> pd.DataFrame:
    return pd.DataFrame(
        {"VaR Absolute": [float(-q)],
         "VaR Diff from Mean": [float(mean_value - q)]}
    )

alpha = 0.05

# left-tail quantile of returns
q_n = float(mu_n + sig_n * stats.norm.ppf(alpha))
q_t = float(stats.t.ppf(alpha, df=nu_t, loc=mu_t, scale=sig_t))

var_n = var_metrics_from_quantile(q_n, float(np.mean(x)))
var_t = var_metrics_from_quantile(q_t, float(np.mean(x)))

pd.concat([
    pd.DataFrame([{"Model": "Normal"}]).join(var_n),
    pd.DataFrame([{"Model": "StudentT"}]).join(var_t),
], ignore_index=True)

,Model,VaR Absolute,VaR Diff from Mean
0,Normal,0.052719,0.052188
1,StudentT,0.051901,0.051369


In [14]:
# Problem 1 (d) alpha = 0.05 ES
alpha = 0.01
def es_loss_from_normal_fit(r: np.ndarray, alpha: float) -> float:
    mu = float(np.mean(r))
    sigma = float(np.std(r, ddof=1))
    z = stats.norm.ppf(alpha)
    phi = stats.norm.pdf(z)
    cond_mean = mu - sigma * (phi / alpha)
    return -float(cond_mean)

def fit_student_t(r: np.ndarray):
    df, loc, scale = stats.t.fit(r)
    return float(df), float(loc), float(scale)

def es_loss_from_t_fit(r: np.ndarray, alpha: float):
    df, loc, scale = fit_student_t(r)
    if df <= 1:
        raise ValueError(f"Fitted df={df:.6f} <= 1, ES not defined.")
    c = stats.t.ppf(alpha, df)
    f = stats.t.pdf(c, df)
    F = stats.t.cdf(c, df)
    cond_mean_std = -((df + c**2) / (df - 1.0)) * (f / F)
    cond_mean = loc + scale * cond_mean_std
    es_loss = -float(cond_mean)
    return df, loc, scale, es_loss

es_n = es_loss_from_normal_fit(x, alpha)
_, _, _, es_t = es_loss_from_t_fit(x, alpha)

pd.DataFrame([
    {"Model": "Normal", "ES (loss)": float(es_n)},
    {"Model": "StudentT", "ES (loss)": float(es_t)},
])

,Model,ES (loss)
0,Normal,0.065977
1,StudentT,0.067714


In [15]:
# Problem 1 (d) alpha = 0.01 VaR
def var_metrics_from_quantile(q: float, mean_value: float) -> pd.DataFrame:
    return pd.DataFrame(
        {"VaR Absolute": [float(-q)],
         "VaR Diff from Mean": [float(mean_value - q)]}
    )

alpha = 0.01

# left-tail quantile of returns
q_n = float(mu_n + sig_n * stats.norm.ppf(alpha))
q_t = float(stats.t.ppf(alpha, df=nu_t, loc=mu_t, scale=sig_t))

var_n = var_metrics_from_quantile(q_n, float(np.mean(x)))
var_t = var_metrics_from_quantile(q_t, float(np.mean(x)))

pd.concat([
    pd.DataFrame([{"Model": "Normal"}]).join(var_n),
    pd.DataFrame([{"Model": "StudentT"}]).join(var_t),
], ignore_index=True)

,Model,VaR Absolute,VaR Diff from Mean
0,Normal,0.074342,0.073810
1,StudentT,0.077320,0.076788


In [16]:
# Problem 1 (d) alpha = 0.01 ES
alpha = 0.01
def es_loss_from_normal_fit(r: np.ndarray, alpha: float) -> float:
    mu = float(np.mean(r))
    sigma = float(np.std(r, ddof=1))
    z = stats.norm.ppf(alpha)
    phi = stats.norm.pdf(z)
    cond_mean = mu - sigma * (phi / alpha)
    return -float(cond_mean)

def fit_student_t(r: np.ndarray):
    df, loc, scale = stats.t.fit(r)
    return float(df), float(loc), float(scale)

def es_loss_from_t_fit(r: np.ndarray, alpha: float):
    df, loc, scale = fit_student_t(r)
    if df <= 1:
        raise ValueError(f"Fitted df={df:.6f} <= 1, ES not defined.")
    c = stats.t.ppf(alpha, df)
    f = stats.t.pdf(c, df)
    F = stats.t.cdf(c, df)
    cond_mean_std = -((df + c**2) / (df - 1.0)) * (f / F)
    cond_mean = loc + scale * cond_mean_std
    es_loss = -float(cond_mean)
    return df, loc, scale, es_loss

es_n = es_loss_from_normal_fit(x, alpha)
_, _, _, es_t = es_loss_from_t_fit(x, alpha)

pd.DataFrame([
    {"Model": "Normal", "ES (loss)": float(es_n)},
    {"Model": "StudentT", "ES (loss)": float(es_t)},
])

,Model,ES (loss)
0,Normal,0.085093
1,StudentT,0.092326


In [17]:
# Problem 1 (e) alpha = 0.05 historical VaR & ES
alpha = 0.05
x_ = np.asarray(x, dtype=float)

q_hist = float(np.quantile(x_, alpha))
var_hist_abs = -q_hist
var_hist_diff = float(np.mean(x_) - q_hist)

es_hist = -float(np.mean(x_[x_ <= q_hist]))

pd.DataFrame([{
    "Model": "Historical",
    "q_alpha": q_hist,
    "VaR Absolute": var_hist_abs,
    "VaR Diff from Mean": var_hist_diff,
    "ES (loss)": es_hist
}])

,Model,q_alpha,VaR Absolute,VaR Diff from Mean,ES (loss)
0,Historical,-0.053356,0.053356,0.052824,0.071073


In [18]:
# Problem 1 (e) alpha = 0.01 historical VaR & ES
alpha = 0.01
x_ = np.asarray(x, dtype=float)

q_hist = float(np.quantile(x_, alpha))
var_hist_abs = -q_hist
var_hist_diff = float(np.mean(x_) - q_hist)

es_hist = -float(np.mean(x_[x_ <= q_hist]))

pd.DataFrame([{
    "Model": "Historical",
    "q_alpha": q_hist,
    "VaR Absolute": var_hist_abs,
    "VaR Diff from Mean": var_hist_diff,
    "ES (loss)": es_hist
}])

,Model,q_alpha,VaR Absolute,VaR Diff from Mean,ES (loss)
0,Historical,-0.075974,0.075974,0.075442,0.09905


In [20]:
# Problem 1 (f)
discussion = f"""
Based on the result, VaR under both distribution are almost the same value no matter which alpha value we choose.
However ES under Student-t is always larger than under Normal if the data are heavy-tailed.
Since Student-t has heavier tails, so it assigns more probability to extreme negative returns.
ES is more sensitive to tail heaviness than VaR, so differences often show up more strongly in ES.
And the ES under Student-t distribution is also closer to the result of historical imulation.
All above indicate that fitting a Student-t model to the data is more properly than a normal distribution.
""".strip()

print(discussion)

Based on the result, VaR under both distribution are almost the same value no matter which alpha value we choose.
However ES under Student-t is often larger than under Normal if the data are heavy-tailed.
Since Student-t has heavier tails, so it assigns more probability to extreme negative returns.
ES is more sensitive to tail heaviness than VaR, so differences often show up more strongly in ES.


In [ ]:
# Problem 2 (a)
Answer = """
In practice, correlations often change faster than volatilities.
So I will use a lower lambda (more responsive) for correlation, and a higher lambda (smoother) for variances, 
so that we can capture the changing correlation structure more quickly, while still maintaining stable variance estimates.
then combine them into a covariance matrix that is both responsive and stable.
""".strip()

print(Answer)

In practice, correlations often change faster than volatilities.
So you might use a lower lambda (more responsive) for correlation, and a higher lambda (smoother) for variances,
then combine them into a covariance matrix that is both responsive and stable.


In [ ]:
# Problem 2 (b)
P2 = f"{BASE_DIR}/problem2.csv"
df2 = pd.read_csv(P2)
df2.head()

,x1,x2,x3,x4,x5
0,-0.011904,-0.012926,-0.006511,-0.006156,0.009584
1,0.021849,0.037438,-0.004951,-0.037799,-0.002164
2,-0.004584,0.016381,0.006534,-0.007579,-0.001832
3,0.006344,0.013604,-0.008160,-0.003432,0.008191
4,-0.025164,-0.037081,-0.001121,0.007057,0.027030


In [23]:
df2_num = df2.select_dtypes(include=[np.number]).copy()
df2_num.shape, df2_num.columns

((1000, 5), Index(['x1', 'x2', 'x3', 'x4', 'x5'], dtype='object'))

In [24]:
# Problem 2 (b)
def ew_weights(n: int, lam: float) -> np.ndarray:
    #
    w = (1.0 - lam) * lam ** np.arange(n - 1, -1, -1)
    return w / w.sum()

def ew_cov(df: pd.DataFrame, lam: float) -> pd.DataFrame:
    cols = list(df.columns)
    X = df.to_numpy(dtype=float)
    n = X.shape[0]
    w = ew_weights(n, lam)

    mu = np.sum(X * w[:, None], axis=0)
    Xc = X - mu[None, :]
    S = (Xc * w[:, None]).T @ Xc

    return pd.DataFrame(S, index=cols, columns=cols)

def cov_to_corr(cov: pd.DataFrame) -> pd.DataFrame:
    cols = list(cov.columns)
    v = np.diag(cov.to_numpy(dtype=float))
    sd = np.sqrt(v)

    denom = np.outer(sd, sd)
    with np.errstate(divide="ignore", invalid="ignore"):
        R = cov.to_numpy(dtype=float) / denom

    R = 0.5 * (R + R.T)
    for i in range(len(cols)):
        R[i, i] = 1.0 if (np.isfinite(sd[i]) and sd[i] > 0) else np.nan

    return pd.DataFrame(R, index=cols, columns=cols)

In [26]:
#Problem (b)
cov_094 = ew_cov(df2_num, lam=0.94)
corr_094 = cov_to_corr(cov_094)

corr_094

,x1,x2,x3,x4,x5
x1,1.000000,0.577652,-0.323920,-0.177924,-0.363249
x2,0.577652,1.000000,0.249123,-0.345214,-0.314785
x3,-0.323920,0.249123,1.000000,0.043394,0.353922
x4,-0.177924,-0.345214,0.043394,1.000000,-0.322211
x5,-0.363249,-0.314785,0.353922,-0.322211,1.000000


In [28]:
# Problem 2 (c)
cov_097 = ew_cov(df2_num, lam=0.97)
var_097 = np.diag(cov_097.to_numpy(dtype=float))
print(cov_097)


          x1        x2        x3        x4        x5
x1  0.000338  0.000206 -0.000070 -0.000057 -0.000060
x2  0.000206  0.000501  0.000096 -0.000145 -0.000057
x3 -0.000070  0.000096  0.000189 -0.000020  0.000065
x4 -0.000057 -0.000145 -0.000020  0.000403 -0.000093
x5 -0.000060 -0.000057  0.000065 -0.000093  0.000129


In [ ]:
# Problem 2 (d)
sd_097 = np.sqrt(var_097)
D = np.diag(sd_097)

cov_combo = pd.DataFrame(
    D @ corr_094.to_numpy(dtype=float) @ D,
    index=df2_num.columns,
    columns=df2_num.columns)

cov_combo


,x1,x2,x3,x4,x5
x1,0.000338,0.000238,-0.000082,-0.000066,-0.000076
x2,0.000238,0.000501,0.000077,-0.000155,-0.000080
x3,-0.000082,0.000077,0.000189,0.000012,0.000055
x4,-0.000066,-0.000155,0.000012,0.000403,-0.000074
x5,-0.000076,-0.000080,0.000055,-0.000074,0.000129


In [30]:
# Problem 3 (a)
P3 = f"{BASE_DIR}/problem3.csv"

def read_matrix_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    has_label_col = df.shape[1] >= 2 and not np.issubdtype(df.iloc[:, 0].dtype, np.number)
    if has_label_col:
        idx = df.iloc[:, 0].astype(str)
        mat = df.iloc[:, 1:].astype(float)
        mat.index = idx
        return mat
    return df.astype(float)

M = read_matrix_csv(P3)
print("type(M):", type(M))
print("M shape:", M.shape)
print("M dtypes:", M.dtypes.unique())
M.head()

type(M): <class 'pandas.core.frame.DataFrame'>
M shape: (5, 5)
M dtypes: [dtype('float64')]


,x1,x2,x3,x4,x5
0,0.000338,0.000384,-0.000078,-0.000043,-0.000046
1,0.000384,0.000501,0.000014,-0.000163,0.000066
2,-0.000078,0.000014,0.000189,-0.000101,-0.000050
3,-0.000043,-0.000163,-0.000101,0.000403,-0.000029
4,-0.000046,0.000066,-0.000050,-0.000029,0.000129


In [34]:
# Problem 3 (a)

def pd_status(mat: pd.DataFrame, eps: float = 1e-10) -> str:
    A = mat.to_numpy(dtype=float)
    A = 0.5 * (A + A.T)
    eigs = np.linalg.eigvalsh(A)
    print(eigs)
    m = float(np.min(eigs))
    if m > eps:
        return "Positive Definite (PD)"
    if m >= -eps:
        return "Positive Semi-Definite (PSD)"
    return "Indefinite (Non-Definite)"

status_before = pd_status(M)
status_before

print("This is a Non-Definite matrix with the smallest eigenvalue under zero")

[-4.56160014e-05  1.23566772e-04  1.90829637e-04  4.26728160e-04
  8.64915610e-04]
This is a Non-Definite matrix with the smallest eigenvalue under zero


In [47]:
# Problem 3 (b)

def proj_psd(A: np.ndarray) -> np.ndarray:
    A = 0.5 * (A + A.T)
    w, V = np.linalg.eigh(A)
    w = np.maximum(w, 0.0)
    B = (V * w) @ V.T
    return 0.5 * (B + B.T)

def higham_near_psd_with_fixed_diag(A: np.ndarray, diag_target: np.ndarray, tol: float = 1e-12, maxiter: int = 2000) -> np.ndarray:
    Y = A.copy()
    deltaS = np.zeros_like(A)
    normA = np.linalg.norm(A, ord="fro") + 1e-15

    for _ in range(maxiter):
        R = Y - deltaS
        X = proj_psd(R)
        deltaS = X - R

        Y_new = X.copy()
        np.fill_diagonal(Y_new, diag_target)

        if np.linalg.norm(Y_new - Y, ord="fro") / normA < tol:
            Y = Y_new
            break
        Y = Y_new

    X_final = proj_psd(Y)
    np.fill_diagonal(X_final, diag_target)
    return 0.5 * (X_final + X_final.T)

def higham_cov(cov: pd.DataFrame) -> pd.DataFrame:
    A = cov.to_numpy(dtype=float)
    diag_target = np.diag(A).copy()
    B = higham_near_psd_with_fixed_diag(A, diag_target=diag_target)
    return pd.DataFrame(B, index=cov.index, columns=cov.columns)

M_fixed = higham_cov(M)
status_after = pd_status(M_fixed)
status_before, status_after

[-6.30502768e-17  1.13162820e-04  1.80226681e-04  4.21710588e-04
  8.45324088e-04]


('Indefinite (Non-Definite)', 'Positive Semi-Definite (PSD)')

In [ ]:
# Problem 3 (b)
print("Higham method is an iterative algorithm that will alternate projection until the norm converges (Frbenius norm). \n" 
"During the process the eigenvalue system with negative values will be set to zero, \n" 
"and the algorithm will update the 𝑌 matrix until we max out our iterations, or the weighted norm converges to a set tolerance.")
M_fixed

Higham is an iterative algorithm that will alternate projection until the norm converges (Frbenius norm). 
During the process the eigenvalue system with negative values will be set to zero, 
and the algorithm will update the 𝑌 matrix until we max out our iterations, or the weighted norm converges to a set tolerance.


,x1,x2,x3,x4,x5
0,0.000338,0.000361,-0.000065,-0.000044,-0.000027
1,0.000361,0.000501,0.000003,-0.000162,0.000051
2,-0.000065,0.000003,0.000189,-0.000102,-0.000041
3,-0.000044,-0.000162,-0.000102,0.000403,-0.000030
4,-0.000027,0.000051,-0.000041,-0.000030,0.000129


In [49]:
# Problem 3 (c)

def proj_psd(A: np.ndarray) -> np.ndarray:
    A = 0.5 * (A + A.T)
    w, V = np.linalg.eigh(A)
    w = np.maximum(w, 0.0)
    B = (V * w) @ V.T
    return 0.5 * (B + B.T)

def near_psd_cov(cov: pd.DataFrame) -> pd.DataFrame:
    """Eigenvalue-clipped PSD approximation of a covariance matrix."""
    B = proj_psd(cov.to_numpy(dtype=float))
    return pd.DataFrame(B, index=cov.index, columns=cov.columns)

M_fixed2 = near_psd_cov(M)
status_after = pd_status(M_fixed2)
status_after

[-4.70571787e-20  1.23566772e-04  1.90829637e-04  4.26728160e-04
  8.64915610e-04]


'Positive Semi-Definite (PSD)'

In [ ]:
# Problem 3 (c)
print("First, this method will decompose the correlation matrix. \n"
"Then, it construct a new eigenvector such that all eigenvalues are greater than or equal to 0. \n"
"Third, construct the diagonal scaling matrix and let the square root of the new generated eigenvector to scale the covariance matrix to get the final result.")
M_fixed2

First, this method will decompose the correlation matrix. 
Then, construct a new eigenvector such that all eigenvalues are greater than or equal to 0. 
Third, construct the diagonal scaling matrix and let the square root of the new generated eigenvector to scale the covariance matrix


,x1,x2,x3,x4,x5
0,0.000356,0.000368,-0.000068,-0.000043,-0.000033
1,0.000368,0.000514,0.000006,-0.000162,0.000056
2,-0.000068,0.000006,0.000194,-0.000101,-0.000043
3,-0.000043,-0.000162,-0.000101,0.000403,-0.000030
4,-0.000033,0.000056,-0.000043,-0.000030,0.000139


In [91]:
# Problem 3 (d)
print("I prefer Higham method. Because it has smaller eigenvalue, \n"
"also provides a more accurate approximation to the original matrix, and it is more stable than eigenvalue clipping method.")


I prefer Higham method. Because it has smaller eigenvalue, 
also provides a more accurate approximation to the original matrix, and it is more stable than eigenvalue clipping method.


In [76]:
# Problem 4 (a)
P4 = f"{BASE_DIR}/problem4.csv"

prices = pd.read_csv(P4)
prices.head()

SPY_shares = 20000 / 689.42
AAPL_shares = 20000 / 264.58
MSFT_shares = 20000 / 397.23
GOOGL_shares = 20000 / 314.98
BABA_shares = 20000 / 154.45
holdings = [SPY_shares, AAPL_shares, MSFT_shares, GOOGL_shares, BABA_shares]
print(holdings)


[29.009892373299298, 75.59150351500492, 50.348664501674094, 63.4960949901581, 129.49174490126256]


In [60]:
# Problem 4 (b)

px = prices.copy()
if "Date" in px.columns:
    px["Date"] = pd.to_datetime(px["Date"])
    px = px.sort_values("Date").set_index("Date")
px_num = px.select_dtypes(include=[np.number]).copy()
# arithmetic returns
rets = px_num.pct_change().dropna()
print(rets.head())
print(rets.tail())

                 SPY      AAPL      MSFT     GOOGL      BABA
Date                                                        
2022-01-04 -0.000335 -0.012692 -0.017147 -0.004083 -0.006812
2022-01-05 -0.019202 -0.026600 -0.038388 -0.045876  0.013382
2022-01-06 -0.000940 -0.016693 -0.007902 -0.000200  0.045147
2022-01-07 -0.003953  0.000988  0.000510 -0.005303  0.025113
2022-01-10 -0.001244  0.000116  0.000733  0.012060 -0.011632
                 SPY      AAPL      MSFT     GOOGL      BABA
Date                                                        
2026-02-13  0.000705 -0.022733 -0.001294 -0.010615 -0.018900
2026-02-17  0.001613  0.031668 -0.011113 -0.012103 -0.001926
2026-02-18  0.005038  0.001781  0.006904  0.004337  0.002188
2026-02-19 -0.002637 -0.014261 -0.002853 -0.001582 -0.009630
2026-02-20  0.007232  0.015350 -0.003087  0.040053  0.001167


In [66]:
# Problem 4 (c)
rets_dm = rets - rets.mean(axis=0)
fit_rows = []
for c in rets_dm.columns:
    r = rets_dm[c].to_numpy(dtype=float)
    mu = float(np.mean(r))
    var = float(np.std(r, ddof=1))
    fit_rows.append({
        "Stock": c,
        "mu": float(mu),
        "var": float(var),
    })

N_fits = pd.DataFrame(fit_rows)
N_fits

,Stock,mu,var
0,SPY,2.947021e-19,0.011237
1,AAPL,-1.875377e-19,0.017949
2,MSFT,4.822397e-19,0.017164
3,GOOGL,8.037329e-19,0.020395
4,BABA,1.500301e-18,0.032979


In [67]:
# Problem 4 (c)
fit_rows = []
for c in rets_dm.columns:
    r = rets_dm[c].to_numpy(dtype=float)
    nu, loc, scale = stats.t.fit(r)
    fit_rows.append({
        "Stock": c,
        "nu": float(nu),
        "loc": float(loc),
        "scale": float(scale)
    })

t_fits = pd.DataFrame(fit_rows)
t_fits

,Stock,nu,loc,scale
0,SPY,3.664941,0.000279,0.007723
1,AAPL,3.492912,0.000169,0.012190
2,MSFT,4.126322,0.000206,0.012529
3,GOOGL,4.552051,0.000035,0.015416
4,BABA,3.125143,-0.002160,0.020770


In [84]:
# Problem 4 (d)
U = np.zeros((len(rets_dm), len(rets_dm.columns)), dtype=float)
cols = list(rets_dm.columns)

for j, c in enumerate(cols):
    x = rets_dm[c].to_numpy(dtype=float)
    ranks = stats.rankdata(x, method="average")
    U[:, j] = (ranks - 0.5) / len(x)

Z = stats.norm.ppf(U)
corr_spear = rets_dm.corr(method="spearman")
corr_spear

def higham_corr(corr: pd.DataFrame) -> pd.DataFrame:
    A = corr.to_numpy(dtype=float)
    A = 0.5 * (A + A.T)

    def proj_psd(A):
        w, V = np.linalg.eigh(A)
        w = np.maximum(w, 0.0)
        B = (V * w) @ V.T
        return 0.5 * (B + B.T)

    # alternating projections: PSD + set diag=1
    Y = A.copy()
    deltaS = np.zeros_like(A)
    diag_target = np.ones(A.shape[0], dtype=float)
    normA = np.linalg.norm(A, ord="fro") + 1e-15

    for _ in range(2000):
        R = Y - deltaS
        X = proj_psd(R)
        deltaS = X - R
        Y_new = X.copy()
        np.fill_diagonal(Y_new, diag_target)
        if np.linalg.norm(Y_new - Y, ord="fro") / normA < 1e-12:
            Y = Y_new
            break
        Y = Y_new

    B = proj_psd(Y)
    np.fill_diagonal(B, 1.0)

    # rescale to correlation
    d = np.sqrt(np.diag(B))
    Dinv = np.diag(1.0 / d)
    C = Dinv @ B @ Dinv
    C = 0.5 * (C + C.T)
    np.fill_diagonal(C, 1.0)

    return pd.DataFrame(C, index=corr.index, columns=corr.columns)

# Try Cholesky; if fails, fix and retry
try:
    L = np.linalg.cholesky(corr_spear.to_numpy(dtype=float))
    corr_used = corr_spear.copy()
    print("Cholesky OK on Spearman correlation (no fix needed).")
except np.linalg.LinAlgError:
    corr_used = higham_corr(corr_spear)
    L = np.linalg.cholesky(corr_used.to_numpy(dtype=float))
    print("Spearman corr not PD; applied Higham fix, then Cholesky.")

corr_used

Cholesky OK on Spearman correlation (no fix needed).


,SPY,AAPL,MSFT,GOOGL,BABA
SPY,1.000000,0.695532,0.747113,0.678570,0.373796
AAPL,0.695532,1.000000,0.578527,0.564982,0.310127
MSFT,0.747113,0.578527,1.000000,0.631735,0.257240
GOOGL,0.678570,0.564982,0.631735,1.000000,0.323606
BABA,0.373796,0.310127,0.257240,0.323606,1.000000


In [ ]:
# Problem 4 (d)
N_SIMS = 1_000_000
rng = np.random.default_rng(12345)

# simulate correlated normals -> uniforms
z_sim = rng.standard_normal(size=(N_SIMS, len(cols))) @ L.T
u_sim = stats.norm.cdf(z_sim)

# transform uniforms via each stock's fitted t marginal
r_sim = np.zeros_like(u_sim)
fit_map = {row["Stock"]: (row["nu"], row["loc"], row["scale"]) for _, row in t_fits.iterrows()}

for j, c in enumerate(cols):
    nu, loc, scale = fit_map[c]
    r_sim[:, j] = stats.t.ppf(u_sim[:, j], df=nu, loc=loc, scale=scale)

r_sim[:3]

array([[-0.01354404, -0.00090902, -0.02386531, -0.01996849, -0.01369975],
       [-0.00610671, -0.02352363, -0.00296684, -0.00510781, -0.08005009],
       [ 0.03129325,  0.04983341,  0.0202469 ,  0.04991158,  0.0126624 ]])

In [ ]:
# Problem 4 (d)
if "Date" in prices.columns:
    prices_sorted = prices.copy()
    prices_sorted["Date"] = pd.to_datetime(prices_sorted["Date"])
    prices_sorted = prices_sorted.sort_values("Date")
    p0_row = prices_sorted.iloc[0]
else:
    p0_row = prices.iloc[0]

# holdings is a list of shares for each asset in the same order as `cols`
# per-asset pnl and total pnl
pnl_asset = {}
v0_asset = {}

for j, c in enumerate(cols):
    p0 = float(p0_row[c])
    h = holdings[j] # use the holding for specific asset
    v0_asset[c] = h * p0
    p1 = p0 * (1.0 + r_sim[:, j])
    pnl_asset[c] = h * (p1 - p0)

pnl_total = np.zeros(N_SIMS, dtype=float)
for c in cols:
    pnl_total += pnl_asset[c]

v0_total = float(sum(v0_asset.values()))
v0_asset, v0_total

({'SPY': 13108.850428054564,
  'AAPL': 13463.124061749992,
  'MSFT': 16307.850689265348,
  'GOOGL': 9137.355090625793,
  'BABA': 14804.051461903631},
 66821.23173159933)

In [ ]:
# Problem 4 (d)
# historical simulation for total pnl
alpha = 0.01
x_ = np.asarray(x, dtype=float)
q_hist = float(np.quantile(x_, alpha))
var_hist_abs = -q_hist
es_hist = -float(np.mean(x_[x_ <= q_hist]))

pd.DataFrame([{
    "Model": "Historical",
    "q_alpha": q_hist,
    "VaR Absolute": var_hist_abs,
    "ES (loss)": es_hist
}])

# apply the same historical procedure to each stock's returns
hist_rows = []
for c in cols:
    r = rets[c].to_numpy(dtype=float)
    q = float(np.quantile(r, alpha))
    var_abs = -q
    var_diff = float(np.mean(r) - q)
    es = -float(np.mean(r[r <= q]))
    hist_rows.append({
        "Stock": c,
        "q_alpha": q,
        "VaR_his99": var_abs,
        "ES_his99": es
    })

hist_var_tbl = pd.DataFrame(hist_rows)
hist_var_tbl

def var_es_loss_from_pnl(pnl: np.ndarray, alpha: float):
    loss = -pnl
    var = float(np.quantile(loss, 1.0 - alpha))
    es = float(loss[loss >= var].mean())
    return var, es

rows = []
for c in cols:
    var, es = var_es_loss_from_pnl(pnl_asset[c], alpha)
    rows.append({
        "Stock": c,
        "VaR99_$": var,
        "ES99_$": es,
        "VaR99_Pct": var / v0_asset[c],
        "ES99_Pct": es / v0_asset[c],
    })

varT, esT = var_es_loss_from_pnl(pnl_total, alpha)
rows.append({
    "Stock": "Total",
    "VaR99_$": varT,
    "ES99_$": esT,
    "VaR99_Pct": varT / v0_total,
    "ES99_Pct": esT / v0_total,
})

risk_tbl = pd.DataFrame(rows)

final_table = hist_var_tbl.merge(risk_tbl, on="Stock", how="outer")
print(final_table)

   Stock   q_alpha  VaR_his99  ES_his99      VaR99_$       ES99_$  VaR99_Pct  \
0   AAPL -0.048116   0.048116  0.057597   665.935659   964.214183   0.049464   
1   BABA -0.080594   0.080594  0.098660  1392.289112  2107.739852   0.094048   
2  GOOGL -0.049235   0.049235  0.068468   493.019243   669.120299   0.053956   
3   MSFT -0.043155   0.043155  0.057472   749.064598  1035.098359   0.045933   
4    SPY -0.031242   0.031242  0.040442   393.913605   564.950083   0.030049   
5  Total       NaN        NaN       NaN  2689.080597  3632.470039   0.040243   

   ES99_Pct  
0  0.071619  
1  0.142376  
2  0.073229  
3  0.063472  
4  0.043097  
5  0.054361  


In [83]:
print(
    "From the Monte Carlo P&L distribution (loss = −P&L), the 99% VaR is defined as the 99th percentile of losses,\n"
    "and the 99% ES is defined as the average loss conditional on losses exceeding VaR.\n"
    f"In this output, the total portfolio has VaR99 ≈ $2,689 (≈4.02% of initial value) and ES99 ≈ $3,632 (≈5.43%).\n"
    "Across single assets (percent-of-value), BABA has the largest tail risk, while SPY has the smallest."
)

From the Monte Carlo P&L distribution (loss = −P&L), the 99% VaR is defined as the 99th percentile of losses,
and the 99% ES is defined as the average loss conditional on losses exceeding VaR.
In this output, the total portfolio has VaR99 ≈ $2,689 (≈4.02% of initial value) and ES99 ≈ $3,632 (≈5.43%).
Across single assets (percent-of-value), BABA has the largest tail risk, while SPY has the smallest.


In [93]:
print("Comparing to the historical simulation approach, the Monte Carlo method with fitted marginals and Spearman correlation captures the tail risk more effectively, especially for heavy-tailed assets like BABA. \n"
"The historical simulation's VaR and ES are generally lower, likely underestimating the risk due to limited historical data and not fully capturing extreme events. \n"
"Overall, the Monte Carlo approach provides a more comprehensive view of the portfolio's tail risk compared to the historical simulation method.")

Comparing to the historical simulation approach, the Monte Carlo method with fitted marginals and Spearman correlation captures the tail risk more effectively, especially for heavy-tailed assets like BABA. 
The historical simulation's VaR and ES are generally lower, likely underestimating the risk due to limited historical data and not fully capturing extreme events. 
Overall, the Monte Carlo approach provides a more comprehensive view of the portfolio's tail risk compared to the historical simulation method.
